In [2]:
import json
import os

class SeizureSegment:
    """
    Represents a seizure segment with associated metadata.
    """
    def __init__(self, sfreq, high_pass, low_pass, channels, patient, session, segment, montage, total_duration,
                 seizure_duration, percentage, edf_size_kb, data_points, events):
        self.sfreq = float(sfreq)
        self.high_pass = float(high_pass)
        self.low_pass = float(low_pass)
        self.channels = list(channels)
        
        self.patient = str(patient)
        self.session = str(session)
        self.segment = str(segment)
        self.montage = str(montage)
        self.total_duration_sec = float(total_duration)
        self.seizure_duration_sec = float(seizure_duration)
        self.percentage = float(percentage)
        self.edf_size_kb = float(edf_size_kb)
        self.data_points = int(data_points)
        self.events = list(events)

        self.file_path = os.path.join(self.patient, self.session, self.montage, self.segment)
        
    @classmethod
    def from_dict(cls, data):
        """Reconstruct SeizureSegment from a dictionary."""
        events = [{'pair': frozenset(event['pair']), 'start_time': event['start_time'], 'stop_time': event['stop_time'], 'label': event['label']} for event in data['events']]  # 将 list 转换回 frozenset
        obj = cls(
            sfreq=data['sfreq'],
            high_pass=data['high_pass'],
            low_pass=data['low_pass'],
            channels=data['channels'],
            patient=data['patient'],
            session=data['session'],
            segment=data['segment'],
            montage=data['montage'],
            total_duration=data['total_duration_sec'],
            seizure_duration=data['seizure_duration_sec'],
            percentage=data['percentage'],
            edf_size_kb=data['edf_size_kb'],
            data_points=data['data_points'],
            events=events
        )
        return obj

    def __repr__(self):
        return (f"<SeizureSegment patient='{self.patient}' "
                f"session='{self.session}' segment='{self.segment}'>")
        
def load_segments_from_json(file_path):
    """Load seizure segments from a JSON file."""
    with open(file_path, 'r') as f:
        data = json.load(f)
    return [SeizureSegment.from_dict(item) for item in data]

# Load segments
json_file_path = '/home/students/wcao/GNN4TS/wavelet_bi/my_project/data/raw/segments.json'
segments = load_segments_from_json(json_file_path)

In [3]:
# group by patient
from collections import defaultdict
patient_segments = defaultdict(list)
for seg in segments:
    patient_segments[seg.patient].append(seg)

# how many patients
print(f"Total unique patients in loaded segments: {len(patient_segments)}")

Total unique patients in loaded segments: 208


In [4]:
# find the patient with the most sessions
max_sessions_patient = None
max_sessions_count = 0
for patient, segs in patient_segments.items():
    sessions = set(seg.session for seg in segs)
    if len(sessions) > max_sessions_count:
        max_sessions_count = len(sessions)
        max_sessions_patient = patient
print(f"Patient with most sessions: {max_sessions_patient} ({max_sessions_count} sessions)")

# how many segments for that patient
max_sessions_patient_segs = patient_segments[max_sessions_patient]
print(f"Total segments for patient {max_sessions_patient}: {len(max_sessions_patient_segs)}")

Patient with most sessions: aaaaanme (9 sessions)
Total segments for patient aaaaanme: 31


In [5]:
# print all of the segments for that patient
for seg in max_sessions_patient_segs:
    print(seg.file_path)

aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t012
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t002
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t003
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t009
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t006
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t013
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t014
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t004
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t007
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t015
aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t010
aaaaanme/s012_2014/01_tcp_ar/aaaaanme_s012_t000
aaaaanme/s011_2014/01_tcp_ar/aaaaanme_s011_t000
aaaaanme/s006_2014/01_tcp_ar/aaaaanme_s006_t002
aaaaanme/s006_2014/01_tcp_ar/aaaaanme_s006_t021
aaaaanme/s006_2014/01_tcp_ar/aaaaanme_s006_t007
aaaaanme/s006_2014/01_tcp_ar/aaaaanme_s006_t003
aaaaanme/s006_2014/01_tcp_ar/aaaaanme_s006_t001
aaaaanme/s006_2014/01_tcp_ar/aaaaanme_s006_t004
aaaaanme/s007_2014/01_tcp_ar/aaaaanme_s007_t002
aaaaanme/s007_2014/01_tcp_ar/aaaaanme_s0

In [8]:
# find segment: aaaaanme_s010_t012
target_segment = None
for seg in max_sessions_patient_segs:
    if seg.segment == 'aaaaanme_s010_t012':
        target_segment = seg
        break
    
if target_segment:
    print(f"Found target segment: {target_segment.file_path}")

Found target segment: aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t012


In [9]:
for e in target_segment.events:
    print(e)

{'pair': frozenset({'F7', 'FP1'}), 'start_time': 0.0, 'stop_time': 295.1387, 'label': 'bckg'}
{'pair': frozenset({'F7', 'FP1'}), 'start_time': 295.1387, 'stop_time': 395.3607, 'label': 'fnsz'}
{'pair': frozenset({'F7', 'FP1'}), 'start_time': 395.3607, 'stop_time': 693.0, 'label': 'bckg'}
{'pair': frozenset({'F7', 'T3'}), 'start_time': 0.0, 'stop_time': 693.0, 'label': 'bckg'}
{'pair': frozenset({'T5', 'T3'}), 'start_time': 0.0, 'stop_time': 693.0, 'label': 'bckg'}
{'pair': frozenset({'T5', 'O1'}), 'start_time': 0.0, 'stop_time': 693.0, 'label': 'bckg'}
{'pair': frozenset({'F8', 'FP2'}), 'start_time': 0.0, 'stop_time': 307.0411, 'label': 'bckg'}
{'pair': frozenset({'F8', 'FP2'}), 'start_time': 307.0411, 'stop_time': 338.4118, 'label': 'fnsz'}
{'pair': frozenset({'F8', 'FP2'}), 'start_time': 338.4118, 'stop_time': 693.0, 'label': 'bckg'}
{'pair': frozenset({'F8', 'T4'}), 'start_time': 0.0, 'stop_time': 693.0, 'label': 'bckg'}
{'pair': frozenset({'T6', 'T4'}), 'start_time': 0.0, 'stop_tim

In [11]:
# 计算采样点数
time_sec = 0.1387
sampling_rate = 250
time_sec * sampling_rate

34.675